In [ ]:
# Если запускаете в чистой среде — раскомментируйте:
# !pip install -U pandas scikit-learn openpyxl


In [ ]:
import os
import re
import time
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("torch:", torch.__version__)


In [ ]:
# -----------------------------
# CONFIG (строго бинарная классификация)
# -----------------------------
SEED = 42

TRAIN_PATH = "/mnt/data/train.csv" if os.path.exists("/mnt/data/train.csv") else "train.csv"
TEST_PATH  = "/mnt/data/test.csv"  if os.path.exists("/mnt/data/test.csv")  else "test.csv"

ID_COL = "id"
TEXT_COL = "text"
TARGET_COL = "score"   # 0/1

# Word-level
WORD_MAX_VOCAB = 80000
WORD_MIN_FREQ = 2
WORD_MAX_LEN = 220

# Char-level
USE_CHAR_BRANCH = True
CHAR_MAX_LEN = 450

# Training
VAL_SIZE = 0.2
BATCH_SIZE = 256 if torch.cuda.is_available() else 64
EPOCHS = 14
LR = 2e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3
GRAD_CLIP = 1.0

# CUDA / AMP
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
USE_AMP = USE_CUDA  # mixed precision
print("USE_CUDA:", USE_CUDA, "| DEVICE:", DEVICE, "| AMP:", USE_AMP, "| BATCH_SIZE:", BATCH_SIZE)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if USE_CUDA:
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

assert {ID_COL, TEXT_COL, TARGET_COL}.issubset(train_df.columns), f"train columns: {train_df.columns}"
assert {ID_COL, TEXT_COL}.issubset(test_df.columns), f"test columns: {test_df.columns}"

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)
test_df[TEXT_COL] = test_df[TEXT_COL].astype(str)

# строго бинарно: 0/1
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)
u = sorted(train_df[TARGET_COL].unique().tolist())
assert u == [0, 1] or u == [0] or u == [1], f"Ожидаем бинарные метки 0/1, но получили: {u}"

print("train:", train_df.shape, "test:", test_df.shape)
print("label dist:", train_df[TARGET_COL].value_counts(normalize=True).to_dict())


In [ ]:
# Токенизация: слова + знаки препинания (лучше для токсичности)
WORD_TOKEN_RE = re.compile(r"[a-zа-яё]+|\d+|[^\s\w]", flags=re.IGNORECASE)

def word_tokenize(text: str):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return WORD_TOKEN_RE.findall(text)

def char_tokenize(text: str):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return list(text)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df[[ID_COL, TEXT_COL]],
    train_df[TARGET_COL].values,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df[TARGET_COL].values,
)

print("train_split:", X_train.shape, "val_split:", X_val.shape)
print("train dist:", dict(zip(*np.unique(y_train, return_counts=True))))
print("val dist:", dict(zip(*np.unique(y_val, return_counts=True))))


In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

def build_word_vocab(texts, min_freq=2, max_vocab=80000):
    counter = Counter()
    for t in texts:
        counter.update(word_tokenize(t))
    tokens = [tok for tok, f in counter.most_common() if f >= min_freq]
    tokens = tokens[: max(0, max_vocab - 2)]
    stoi = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for i, tok in enumerate(tokens, start=2):
        stoi[tok] = i
    return stoi

word_stoi = build_word_vocab(X_train[TEXT_COL].values, min_freq=WORD_MIN_FREQ, max_vocab=WORD_MAX_VOCAB)
WORD_VOCAB_SIZE = len(word_stoi)
print("WORD_VOCAB_SIZE:", WORD_VOCAB_SIZE)


In [ ]:
CHAR_PAD = "<c_pad>"
CHAR_UNK = "<c_unk>"
CHAR_PAD_ID = 0
CHAR_UNK_ID = 1

def build_char_vocab(texts, max_vocab=600):
    counter = Counter()
    for t in texts:
        counter.update(char_tokenize(t))
    tokens = [ch for ch, _ in counter.most_common()]
    tokens = tokens[: max(0, max_vocab - 2)]
    stoi = {CHAR_PAD: CHAR_PAD_ID, CHAR_UNK: CHAR_UNK_ID}
    for i, ch in enumerate(tokens, start=2):
        stoi[ch] = i
    return stoi

char_stoi = build_char_vocab(X_train[TEXT_COL].values, max_vocab=600)
CHAR_VOCAB_SIZE = len(char_stoi)
print("CHAR_VOCAB_SIZE:", CHAR_VOCAB_SIZE)


In [ ]:
def encode_words(text: str, max_len: int):
    toks = word_tokenize(text)
    ids = [word_stoi.get(t, UNK_ID) for t in toks[:max_len]]
    length = len(ids)
    if length < max_len:
        ids += [PAD_ID] * (max_len - length)
    return torch.tensor(ids, dtype=torch.long), length

def encode_chars(text: str, max_len: int):
    chars = char_tokenize(text)
    ids = [char_stoi.get(c, CHAR_UNK_ID) for c in chars[:max_len]]
    length = len(ids)
    if length < max_len:
        ids += [CHAR_PAD_ID] * (max_len - length)
    return torch.tensor(ids, dtype=torch.long), length


In [ ]:
class ToxicHybridDataset(Dataset):
    def __init__(self, df: pd.DataFrame, labels=None):
        self.df = df.reset_index(drop=True)
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, TEXT_COL]
        w_ids, w_len = encode_words(text, WORD_MAX_LEN)

        if USE_CHAR_BRANCH:
            c_ids, c_len = encode_chars(text, CHAR_MAX_LEN)
        else:
            c_ids, c_len = torch.empty(0, dtype=torch.long), 0

        sample = {
            "word_ids": w_ids,
            "word_len": torch.tensor(w_len, dtype=torch.long),
            "char_ids": c_ids,
            "char_len": torch.tensor(c_len, dtype=torch.long),
            "id": self.df.loc[idx, ID_COL],
        }
        if self.labels is not None:
            sample["label"] = torch.tensor(float(self.labels[idx]), dtype=torch.float32)  # бинарная метка
        return sample

def collate_fn(batch):
    word_ids = torch.stack([b["word_ids"] for b in batch], dim=0)
    word_len = torch.stack([b["word_len"] for b in batch], dim=0)
    ids = [b["id"] for b in batch]

    if USE_CHAR_BRANCH:
        char_ids = torch.stack([b["char_ids"] for b in batch], dim=0)
        char_len = torch.stack([b["char_len"] for b in batch], dim=0)
    else:
        char_ids = torch.empty((len(batch), 0), dtype=torch.long)
        char_len = torch.zeros((len(batch),), dtype=torch.long)

    if "label" in batch[0]:
        labels = torch.stack([b["label"] for b in batch], dim=0)
        return word_ids, word_len, char_ids, char_len, labels, ids
    else:
        return word_ids, word_len, char_ids, char_len, ids

train_loader = DataLoader(ToxicHybridDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=USE_CUDA, collate_fn=collate_fn)
val_loader   = DataLoader(ToxicHybridDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=USE_CUDA, collate_fn=collate_fn)
test_loader  = DataLoader(ToxicHybridDataset(test_df[[ID_COL, TEXT_COL]], None), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=USE_CUDA, collate_fn=collate_fn)

len(train_loader), len(val_loader), len(test_loader)


In [ ]:
class SpatialDropout1D(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        x = x.transpose(1, 2)
        x = F.dropout(x, p=self.p, training=True)
        return x.transpose(1, 2)


class AttentionPooling(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.score = nn.Linear(dim, 1)

    def forward(self, x, mask):
        h = torch.tanh(self.proj(x))
        s = self.score(h).squeeze(-1)
        s = s.masked_fill(mask == 0, torch.finfo(s.dtype).min)  # fp16-safe
        a = torch.softmax(s, dim=1)
        v = torch.sum(x * a.unsqueeze(-1), dim=1)
        return v


class WordTextCNN(nn.Module):
    def __init__(self, embed_dim: int, num_filters: int, kernel_sizes):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])

    def forward(self, x_emb):
        x = x_emb.transpose(1, 2)
        feats = []
        for conv in self.convs:
            h = F.relu(conv(x))
            feats.append(torch.max(h, dim=2).values)
        return torch.cat(feats, dim=1)


class CharCNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_filters: int, kernel_sizes, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=CHAR_PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])

    def forward(self, char_ids):
        x = self.embed(char_ids)
        x = self.dropout(x)
        x = x.transpose(1, 2)
        feats = []
        for conv in self.convs:
            h = F.relu(conv(x))
            feats.append(torch.max(h, dim=2).values)
        return torch.cat(feats, dim=1)


class ToxicHybridNet(nn.Module):
    def __init__(self):
        super().__init__()

        WORD_EMBED_DIM = 160
        WORD_GRU_HIDDEN = 256
        WORD_GRU_LAYERS = 2

        CNN_FILTERS = 160
        CNN_KERNELS = (3, 4, 5)

        CHAR_EMBED_DIM = 32
        CHAR_FILTERS = 128
        CHAR_KERNELS = (3, 5, 7)

        DROPOUT_EMB = 0.2
        DROPOUT = 0.35
        MLP_HIDDEN = 256

        self.word_embed = nn.Embedding(WORD_VOCAB_SIZE, WORD_EMBED_DIM, padding_idx=PAD_ID)
        self.word_sdrop = SpatialDropout1D(DROPOUT_EMB)

        self.word_gru = nn.GRU(
            input_size=WORD_EMBED_DIM,
            hidden_size=WORD_GRU_HIDDEN,
            num_layers=WORD_GRU_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=(DROPOUT if WORD_GRU_LAYERS > 1 else 0.0),
        )
        self.word_attn = AttentionPooling(WORD_GRU_HIDDEN * 2)
        self.word_cnn = WordTextCNN(WORD_EMBED_DIM, CNN_FILTERS, CNN_KERNELS)

        if USE_CHAR_BRANCH:
            self.char_cnn = CharCNN(CHAR_VOCAB_SIZE, CHAR_EMBED_DIM, CHAR_FILTERS, CHAR_KERNELS, dropout=DROPOUT_EMB)
            char_out_dim = CHAR_FILTERS * len(CHAR_KERNELS)
        else:
            self.char_cnn = None
            char_out_dim = 0

        gru_feat_dim = WORD_GRU_HIDDEN * 2
        cnn_feat_dim = CNN_FILTERS * len(CNN_KERNELS)
        self.feat_dim = (gru_feat_dim * 2) + cnn_feat_dim + char_out_dim

        self.bn1 = nn.BatchNorm1d(self.feat_dim)
        self.drop = nn.Dropout(DROPOUT)

        self.fc1 = nn.Linear(self.feat_dim, MLP_HIDDEN)
        self.bn2 = nn.BatchNorm1d(MLP_HIDDEN)
        self.fc2 = nn.Linear(MLP_HIDDEN, 1)  # 1 logit => бинарная классификация

    def forward(self, word_ids, word_len, char_ids=None):
        w = self.word_embed(word_ids)
        w = self.word_sdrop(w)

        cnn_feat = self.word_cnn(w)

        lengths_cpu = word_len.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(w, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.word_gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=word_ids.size(1))

        B, Lw = word_ids.size(0), word_ids.size(1)
        idx = torch.arange(Lw, device=word_ids.device).unsqueeze(0).expand(B, Lw)
        mask = (idx < word_len.unsqueeze(1)).long()

        attn_feat = self.word_attn(out, mask)

        out_masked = out.masked_fill(mask.unsqueeze(-1) == 0, torch.finfo(out.dtype).min)  # fp16-safe
        max_feat = torch.max(out_masked, dim=1).values

        feats = [attn_feat, max_feat, cnn_feat]

        if USE_CHAR_BRANCH and (char_ids is not None):
            feats.append(self.char_cnn(char_ids))

        x = torch.cat(feats, dim=1)
        x = self.bn1(x)
        x = self.drop(x)

        x = F.relu(self.fc1(x))
        x = self.bn2(x)
        x = self.drop(x)

        return self.fc2(x).squeeze(-1)


model = ToxicHybridNet().to(DEVICE)
print("Total params:", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
pos = float(y_train.sum())
neg = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)
print("pos_weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,
    div_factor=10.0,
    final_div_factor=50.0,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_y = [], []

    for word_ids, word_len, char_ids, char_len, labels, ids in loader:
        word_ids = word_ids.to(DEVICE, non_blocking=True)
        word_len = word_len.to(DEVICE, non_blocking=True)
        if USE_CHAR_BRANCH:
            char_ids = char_ids.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model(word_ids, word_len, char_ids if USE_CHAR_BRANCH else None)
        loss = criterion(logits, labels)

        total_loss += loss.item() * word_ids.size(0)
        n += word_ids.size(0)

        all_logits.append(logits.detach().cpu().numpy())
        all_y.append(labels.detach().cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_y = np.concatenate(all_y, axis=0)
    probs = sigmoid_np(all_logits)
    auc = roc_auc_score(all_y, probs)
    return total_loss / n, auc


def train_one_epoch(model, loader):
    model.train()
    total_loss, n = 0.0, 0

    for word_ids, word_len, char_ids, char_len, labels, ids in loader:
        word_ids = word_ids.to(DEVICE, non_blocking=True)
        word_len = word_len.to(DEVICE, non_blocking=True)
        if USE_CHAR_BRANCH:
            char_ids = char_ids.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(word_ids, word_len, char_ids if USE_CHAR_BRANCH else None)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        if GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * word_ids.size(0)
        n += word_ids.size(0)

    return total_loss / n


In [ ]:
best_auc = -1.0
best_state = None
pat_left = PATIENCE
history = []

t0 = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader)
    va_loss, va_auc = evaluate(model, val_loader)

    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, "val_auc": va_auc})
    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_auc={va_auc:.6f}")

    if va_auc > best_auc + 1e-4:
        best_auc = va_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        pat_left = PATIENCE
    else:
        pat_left -= 1
        if pat_left <= 0:
            print("Early stopping.")
            break

t1 = time.perf_counter()
print("Training time (sec):", round(t1 - t0, 2))
print("Best val AUC:", round(best_auc, 6))

pd.DataFrame(history)


In [ ]:
# Predict test -> submission.xlsx
assert best_state is not None
model.load_state_dict(best_state)
model.eval()

all_ids = []
all_scores = []

with torch.no_grad():
    for word_ids, word_len, char_ids, char_len, ids in test_loader:
        word_ids = word_ids.to(DEVICE, non_blocking=True)
        word_len = word_len.to(DEVICE, non_blocking=True)
        if USE_CHAR_BRANCH:
            char_ids = char_ids.to(DEVICE, non_blocking=True)

        logits = model(word_ids, word_len, char_ids if USE_CHAR_BRANCH else None)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_scores.append(probs)
        all_ids.extend(ids)

all_scores = np.concatenate(all_scores, axis=0)
submission = pd.DataFrame({"id": all_ids, "score": all_scores.astype(float)})
submission.head()


In [ ]:
out_path = "submission.xlsx"
submission.to_excel(out_path, index=False)
print("Saved:", out_path)
